# SNPVAR

Add pre-computed SNP heritabilities.

In [1]:
from pyprojroot.here import here
import pandas as pd

In [ ]:
%%bash -s {here()}
here=$1
mkdir $here/data/processed/snpvar

mkdir -p $here/data/processed/snpvar

for sumstats in mdd2024_afr mdd2024_eas mdd2024_his mdd2024_sas mdd2025_eur; do
python $here/vendor/polyfun/extract_snpvar.py \
    --sumstats $here/data/processed/sumstats/${sumstats}_hg19_z.parquet \
    --allow-missing \
    --out $here/data/processed/snpvar/${sumstats}_hg19_snpvar.tsv
done

mkdir: cannot create directory ‘/home/madams23/Projects/cvd-mh-loci/data/processed/snpvar’: File exists


[INFO]  Loading sumstats files...
[INFO]  Done in 45.36 seconds
[INFO]  Loading meta-analyzed per-SNP-h2 files...
[INFO]  Done in 61.67 seconds
[INFO]  Merging sumstats with per-SNP h2 data...
[INFO]  Flipping the Z-sign of 5638707 SNPs that A1 in sumstats = A2 in the per-SNP h2 data
[INFO]  Done in 30.02 seconds
[WARNING]  Not all SNPs in the SNPs file were found in the meta file. Wrote a list of missing SNPs to /home/madams23/Projects/cvd-mh-loci/data/processed/snpvar/mdd2024_afr_h19_snpvar.tsv.miss.gz
[INFO]  Writing output file to /home/madams23/Projects/cvd-mh-loci/data/processed/snpvar/mdd2024_afr_h19_snpvar.tsv
[INFO]  Loading sumstats files...
[INFO]  Done in 21.68 seconds
[INFO]  Loading meta-analyzed per-SNP-h2 files...
[INFO]  Done in 59.48 seconds
[INFO]  Merging sumstats with per-SNP h2 data...
[INFO]  Flipping the Z-sign of 3529546 SNPs that A1 in sumstats = A2 in the per-SNP h2 data
[INFO]  Done in 19.77 seconds
[WARNING]  Not all SNPs in the SNPs file were found in the 

In [2]:
import polars as pl
from pyprojroot.here import here

Add snpvar to hg38 sumstats.

In [ ]:
for dataset in ["mdd2024_afr", "mdd2024_eas", "mdd2024_his", "mdd2025_eur", "mdd2024_sas"]:

    sumstats = pl.scan_parquet(here(f"data/processed/sumstats/{dataset}_hg38_z.parquet"))
    snpvar = pl.scan_csv(here(f"data/processed/snpvar/{dataset}_hg19_snpvar.tsv"), separator="\t")

    snpvar_hg38 = (sumstats
        .join(snpvar, on = "SNP", how = "inner", suffix="_hg19")
        .select(
            pl.col("CHR"),
            pl.col("BP"),
            pl.col("SNP"),
            pl.col("A1"),
            pl.col("A2"),
            pl.col("Z"),
            pl.col("SNPVAR")
        )
        .sort(by = ["CHR", "BP"])
    )

    snpvar_hg38.sink_csv(here(f"data/processed/snpvar/{dataset}_hg38_snpvar.tsv"), separator = "\t")
    snpvar_hg38.sink_parquet(here(f"data/processed/snpvar/{dataset}_hg38_snpvar.parquet"))